# DSP data quality check

**Input:**
- `data/interim/dsp.csv` — monthly reports from audio and video platforms (DSPs), 01-2024 → 09-2026. Two report formats: the old one until 06-2026 and a new one from 07-2026.
- `data/reference/releases.csv` — release list of the catalogue (artist, track, ISRC, UPC, release date).
- `data/reference/platforms.csv` — platform list: platform name in the reports → partner name and platform type.

**Goal:** find data problems *before* cleaning. This notebook only looks at the data and changes nothing; the fixes go to `src/clean_data.py`.

A summary table is at the end.

In [1]:
import pandas as pd

# upc is read as text: it is a code, not a number
data = pd.read_csv(
    "../data/interim/dsp.csv",
    parse_dates=["report_month"],
    dtype={"upc": str},
    keep_default_na=False,
    na_values=[""],
)
releases = pd.read_csv("../data/reference/releases.csv", dtype={"upc": str})
platforms = pd.read_csv("../data/reference/platforms.csv")

## 1. Overview

**Check:** table size and first rows.

In [ ]:
# Size of the table (rows, columns) and a first look at the data
print(data.shape)
data.head()

(163766, 8)


**Found:** 163,766 rows, 8 columns.

**Decision:** no action needed.

## 2. Column types

In [3]:
data.dtypes

artist                     str
track                      str
isrc                       str
upc                        str
platform                   str
country                    str
revenue                float64
report_month    datetime64[us]
dtype: object

**Found:** `revenue` is a float, `report_month` is a datetime, the other columns are text, including the codes `isrc` (track) and `upc` (release).

**Decision:** no action needed.

## 3. Missing values

**Check:** are there empty cells?

In [ ]:
data.isna().sum()

**Found:**
- `country` is empty in 37 rows.
- `isrc` is empty in 2 rows.
- All other columns have no missing values.

**Decision:**
- `country`: keep the rows — the revenue was really paid. In `clean_data.py` fill the missing `country` with `Unknown`, so totals by country match the overall total.
- `isrc`: in `clean_data.py`, fill it from the release list `data/reference/releases.csv` (matched by `artist` + `track`).

## 4. Negative and zero revenue

**Check:** are there rows with negative or zero revenue?

In [5]:
print("Negative revenue:", (data["revenue"] < 0).sum())
print("Zero revenue:", (data["revenue"] == 0).sum())

Negative revenue: 89
Zero revenue: 47


**Found:**
- 89 rows with negative revenue.
- 47 rows with exactly zero revenue.

**Decision:**
- Negative rows: keep. They are platform corrections (clawbacks of earlier payments); payouts to artists were calculated net of them, so they stay to keep totals matching the payouts.
- Zero rows: drop in `clean_data.py`. They will not be needed anywhere.

## 5. Duplicate rows

In [6]:
data.duplicated().sum()

np.int64(6370)

**Found:** 6,370 rows are fully identical to another row on all 8 columns.

**Decision:** keep all rows. Identical sales are normal: the same track can be sold several times in the same country and month for the same price, and the reports have no transaction ID to tell these sales apart. They are part of the report total that was paid out.

## 6. Artists, tracks and codes vs release list

**Check:** every row matches the release list: rows with ISRC by `artist` + `track` + `isrc` + `upc`, rows without ISRC (section 3) by `artist` + `track` + `upc`. A row with a different spelling or a wrong code would show up as "not found".
- **ISRC** — code of a recording (track); **UPC** — code of a release (single, album).

In [7]:
data[["artist", "track"]].nunique()

artist    13
track     22
dtype: int64

In [8]:
# Compare every row with the release list:
# rows with ISRC by artist + track + isrc + upc, rows without ISRC by artist + track + upc
full_key = ["artist", "track", "isrc", "upc"]
short_key = ["artist", "track", "upc"]

has_isrc = data["isrc"].notna()
with_isrc = data.loc[has_isrc, full_key].merge(releases[full_key], how="left", indicator=True)
without_isrc = data.loc[~has_isrc, short_key].merge(releases[short_key], how="left", indicator=True)

print("Rows with ISRC:", len(with_isrc), "| not found:", (with_isrc["_merge"] == "left_only").sum())
print("Rows without ISRC:", len(without_isrc), "| not found:", (without_isrc["_merge"] == "left_only").sum())

Rows with ISRC: 163764 | not found: 499
Rows without ISRC: 2 | not found: 0


In [9]:
# Rows not found: do they match a release on everything except the artist (track + isrc + upc)?
not_found = with_isrc[with_isrc["_merge"] == "left_only"]
no_artist_key = ["track", "isrc", "upc"]
check_rest = not_found[no_artist_key].merge(releases[no_artist_key], how="left", indicator=True)
print("Not found rows:", len(not_found))
print("Not found rows that match the release list on track + isrc + upc:", (check_rest["_merge"] == "both").sum())
not_found.groupby(["artist", "track"]).size()

Not found rows: 499
Not found rows that match the release list on track + isrc + upc: 499


artist                   track       
ECSTVZZZ                 KAZAKH STYLE     70
ECSTVZZZ and Beno        KAZAKH STYLE    353
ECSTVZZZ featuring Beno  KAZAKH STYLE     76
dtype: int64

**Found:**
- 13 artist spellings, 22 tracks.
- Both rows without ISRC match the release list on `artist` + `track` + `upc`.
- Of the rows with ISRC, 499 are not found. All of them are **one collaboration track** whose artist is written in 3 other ways than in the release list (`X`, `X and Y`, `X featuring Y` instead of `X, Y`). Their `track`, `isrc` and `upc` match one release in the release list → only the artist spelling differs.
- All other rows match the release list on `artist` + `track` + `isrc` + `upc`.

**Decision:**
- In `clean_data.py`, take `artist` from the release list by `isrc`: the collaboration track gets one credit, as in the release list (so there are 10 artists, not 13).
- No re-check is needed after cleaning: the filled `isrc` and the replaced `artist` are taken from the release list itself.
- `isrc` is the track key in the analysis (industry standard, does not depend on spelling). It is used only inside the project and is not shown in public outputs. `upc` is kept as in the reports.

## 7. Platforms

**Check:** every platform name from the reports is in the platform list. The same partner can appear under several names: different spelling, a sub-service, a region, or a new name in the new report format.

In [10]:
# Platform name in the reports -> partner name from the platform list
# (a preview: the data itself is changed in clean_data.py)
name_to_partner = dict(zip(platforms["report_name"], platforms["platform"]))
new_names = sorted(set(data["platform"]) - set(name_to_partner))

if new_names:
    print("New platform names found:", new_names)
else:
    partner = data["platform"].map(name_to_partner)
    print("All", data["platform"].nunique(), "platform names are in the platform list")
    print("Partners after replacement:", partner.nunique())

All 46 platform names are in the platform list
Partners after replacement: 32


**Found:**
- 46 platform names in the reports, all of them are in the platform list → 32 partners.

**Decision:**
- In `clean_data.py`, replace `platform` with the partner name from `data/reference/platforms.csv` (matched by `report_name`). An `assert` checks that every row has a partner, so a new name in a future report stops the script.
- `platform` = the partner that paid: a service or a company (for example, `Meta (Instagram, Facebook)`).

## 8. Country codes

**Check:** every country is a 2-letter ISO code (for example `US`, `DE`). Missing values are covered in section 3.

In [ ]:
# ISO country code = exactly 2 capital letters
codes = data["country"].dropna()
is_iso = codes.str.fullmatch(r"[A-Z]{2}")
print("Unique country codes:", codes.nunique())
print("Codes not in ISO format:", (~is_iso).sum())

**Found:** 237 different country codes, all in the 2-letter ISO format.

**Decision:** no action needed.

## 9. Rows and revenue per month

**Check:** does the number of rows or the revenue jump unexpectedly from month to month?

In [ ]:
# Rows and total revenue in each monthly report
monthly = data.groupby("report_month").agg(
    rows=("revenue", "size"),
    revenue=("revenue", "sum"),
)
monthly

**Found:**
- The number of rows jumps from month to month: very high in 11-2024 and 05-2025, very low in 04-2025.
- Revenue drops sharply from 12-2024 and keeps falling through 2025–2026.
- **06-2026** (the last old-format report) has almost zero revenue. From **07-2026** the new format starts with a small but stable revenue.

**Decision:**
- Row jumps and revenue drop → real data, kept as is: revenue per month is what was paid. They will be analysed in the analysis notebooks.
- 06–09 2026 → real data, kept as is. The contract with the old distributor was terminated, and the catalogue was moved to a new one (new report format from 07-2026). In the analysis notebooks these months will be treated as a wind-down period: marked on charts and not compared directly with earlier months.

## Summary

| # | Check | Found | Decision |
|---|---|---|---|
| 1 | Overview | 163,766 rows, 8 columns | OK |
| 2 | Column types | as expected | no action |
| 3 | Missing values | 37 rows without `country`, 2 without `isrc` | fill `country` with `Unknown`; fill `isrc` from `releases.csv` |
| 4 | Negative / zero revenue | 89 negative, 47 zero | keep negative (clawbacks), drop zero |
| 5 | Duplicate rows | 6,370 identical rows | keep — separate sales, paid out |
| 6 | Artists, tracks, codes vs release list | 13 / 22 values; 499 rows of one collaboration track with another artist spelling, codes match | take `artist` from `releases.csv` by `isrc`; `isrc` = track key |
| 7 | Platforms | 46 names, all in `platforms.csv` → 32 partners | replace with the partner from `platforms.csv` |
| 8 | Country codes | 237 codes, all ISO | no action |
| 9 | Rows per month | row jumps; revenue drop from 12-2024; 06-2026 ≈ 0, new format from 07-2026 | keep all; analyse the jumps and the drop separately; treat 06–09 2026 as wind-down period |

**Next step:** `src/clean_data.py`.